# Day 8 - One last example: a CNN + LSTM for heartbeats

Everything from the course in one notebook - with the **same recipe** as always:
1. **look at the data**
2. **prepare** it: tensors, train / validation / test
3. **the model**: a class `CNN_LSTM` - a CNN finds short patterns, an LSTM reads them in order
4. **the training loop**
5. **evaluate** - the test set only once, at the very end

A **CPU runtime is enough** (training takes about 1 minute). Run the cells from top to bottom.

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, balanced_accuracy_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(0)

Download the data: **ECG5000** - 5 000 heartbeats from one long ECG, each beat cut to **140 time steps**
(the same data as in Lab 5).

In [ ]:
!wget -q https://www.timeseriesclassification.com/aeon-toolkit/ECG5000.zip
!unzip -q -o ECG5000.zip

## 1. Look at the data
Every row of the file is one heartbeat: the **first number is the class** (1 ... 5), then come the 140 values.

In [ ]:
data = np.concatenate([np.loadtxt("ECG5000_TRAIN.txt"), np.loadtxt("ECG5000_TEST.txt")])
print("table:", data.shape)                 # (5000, 141): 5000 beats, 1 label + 140 values

signals = data[:, 1:]                        # (5000, 140)
labels = data[:, 0].astype(int) - 1          # classes 0 ... 4 (PyTorch starts counting at 0)

names = ["normal", "R-on-T PVC", "PVC", "SP", "unclassified"]
for c in range(5):
    print(c, names[c].ljust(13), (labels == c).sum(), "beats")

Very **imbalanced**: 58 % normal beats, but only 24 "unclassified" ones - like DermaMNIST in the challenge.
Let us look at 10 beats of every class:

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(16, 3), sharey=True)
for c in range(5):
    axes[c].plot(signals[labels == c][:10].T, color="#2E86AB", alpha=0.5)
    axes[c].set_title(names[c])
    axes[c].set_xlabel("time step")
plt.show()

## 2. Prepare the data
- **split** 70 / 15 / 15 % into training, validation and test - *stratified* (same share of every class in each part)
- **tensors** in the shape a `Conv1d` wants: (beats, channels, time steps) = (N, **1**, 140) - one channel, like one
  ECG lead

In [ ]:
X = torch.tensor(signals, dtype=torch.float32).unsqueeze(1)     # (5000, 1, 140)
y = torch.tensor(labels)                                         # (5000,)

X_train, X_rest, y_train, y_rest = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
X_val, X_test, y_val, y_test = train_test_split(X_rest, y_rest, test_size=0.5, stratify=y_rest, random_state=0)
print("training:", X_train.shape, " validation:", X_val.shape, " test:", X_test.shape)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)

## 3. The model: `CNN_LSTM`

| step | layer | what it does | shape after it |
|---|---|---|---|
| input | | one heartbeat | (batch, 1, 140) |
| 1 | `Conv1d` + ReLU + max pool | 16 small filters (5 time steps) find short patterns: peaks, slopes | (batch, 16, 70) |
| 2 | `Conv1d` + ReLU + max pool | 32 filters combine them to bigger patterns | (batch, 32, 35) |
| 3 | `permute` | turn it around: 35 **time steps** with 32 numbers each | (batch, 35, 32) |
| 4 | `LSTM` | reads the 35 steps **in order** and keeps a memory of 64 numbers | memory: (batch, 64) |
| 5 | `Linear` | memory after the last step → one score per class | (batch, 5) |

The CNN (day 4) is good at **short, local** patterns. The LSTM (day 5) is good at the **order** of things.
Together: the CNN makes the sequence shorter (140 → 35), the LSTM reads it.

In [ ]:
class CNN_LSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=16, kernel_size=5, padding=2)
        self.conv2 = nn.Conv1d(in_channels=16, out_channels=32, kernel_size=5, padding=2)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(kernel_size=2)          # halves the length
        self.lstm = nn.LSTM(input_size=32, hidden_size=64, batch_first=True)
        self.out = nn.Linear(64, 5)

    def forward(self, x):                      # x: (batch, 1, 140)
        x = self.pool(self.relu(self.conv1(x)))         # (batch, 16, 70)
        x = self.pool(self.relu(self.conv2(x)))         # (batch, 32, 35)
        x = x.permute(0, 2, 1)                          # (batch, 35, 32): time steps first
        outputs, (h, c) = self.lstm(x)                  # h: memory after the last time step, (1, batch, 64)
        return self.out(h[0])                           # (batch, 5): one score per class


model = CNN_LSTM()
print(model)
print("parameters:", sum(p.numel() for p in model.parameters()))

Before training: send 8 beats through the model and check the shape. **Always do this first** - most errors are
shape errors.

In [ ]:
scores = model(X_train[:8])
print("input:", X_train[:8].shape, " → output:", scores.shape)    # (8, 1, 140) → (8, 5)

## 4. The training loop
The recipe from day 2 - the same for MLP, CNN, LSTM, Transformer:
**prediction → loss → gradients → step**.

Because the classes are imbalanced, rare classes get a **larger weight** in the loss
(weight = number of beats / (5 · beats of this class)) - the same idea as in the challenge.

In [ ]:
weights = len(y_train) / (5 * torch.bincount(y_train))
print("class weights:", weights.round(decimals=2))

loss_fn = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.Adam(model.parameters(), lr=0.003)

EPOCHS = 15
train_losses, val_scores = [], []

for epoch in range(EPOCHS):
    model.train()
    total = 0
    for xb, yb in train_loader:
        pred = model(xb)                  # 1. prediction
        loss = loss_fn(pred, yb)          # 2. loss
        optimizer.zero_grad()             # 3. delete the old gradients
        loss.backward()                   # 4. new gradients (backprop)
        optimizer.step()                  # 5. one step downhill
        total += loss.item() * len(yb)

    model.eval()                          # check on the validation beats - no training here
    with torch.no_grad():
        val_pred = model(X_val).argmax(dim=1)
    val_bal = balanced_accuracy_score(y_val, val_pred)

    train_losses.append(total / len(y_train))
    val_scores.append(val_bal)
    print(f"epoch {epoch + 1:2d}: training loss {train_losses[-1]:.3f}   validation balanced accuracy {val_bal:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].plot(range(1, EPOCHS + 1), train_losses, marker="o")
axes[0].set_title("training loss"); axes[0].set_xlabel("epoch")
axes[1].plot(range(1, EPOCHS + 1), val_scores, marker="o", color="#EB811B")
axes[1].set_title("validation balanced accuracy"); axes[1].set_xlabel("epoch")
plt.show()

## 5. Evaluate on the test set - once, at the very end

In [ ]:
model.eval()
with torch.no_grad():
    test_pred = model(X_test).argmax(dim=1)

print("accuracy:         ", round(accuracy_score(y_test, test_pred), 3))
print("balanced accuracy:", round(balanced_accuracy_score(y_test, test_pred), 3))

ConfusionMatrixDisplay.from_predictions(y_test, test_pred, display_labels=names, normalize="true",
                                        values_format=".2f", xticks_rotation=30, cmap="Blues")
plt.show()

<details><summary>👀 What you should see</summary>

- the training loss goes down, the validation balanced accuracy goes up and then wobbles
- accuracy on the test set about 0.88, balanced accuracy only about 0.6 - the rare classes are hard
- in the confusion matrix: "normal" almost always right (~0.98), "R-on-T PVC" mostly (~0.8); "PVC", "SP" and
  "unclassified" only about half or less - the test set has only 15, 29 and **3** of these beats, so these numbers
  jump a lot from run to run
</details>

## 🔧 Try it yourself
1. **Only the CNN**: in `forward`, replace the LSTM lines by `x = x.mean(dim=2)` and `return self.out(x)` - and
   change the last layer to `nn.Linear(32, 5)`. Better or worse?
2. **Without class weights**: `loss_fn = nn.CrossEntropyLoss()`. What happens to the balanced accuracy?
3. **A GRU** instead of the LSTM: `nn.GRU(...)` - and `outputs, h = self.lstm(x)` (a GRU has no `c`)